# Data Scientist — Auto-Approve Demo

This notebook submits a job to a **Data Owner** who has auto-approval enabled.

**Prerequisites:** The DO should have already run their notebook and registered the approved script.

## 1. Install

In [ ]:
!pip install -q syft-client

## 2. Configuration

In [ ]:
DO_EMAIL = "your-do-email@example.com"   # Data Owner's email
DS_EMAIL = "your-ds-email@example.com"   # Your email (Data Scientist)

## 3. Login as Data Scientist

In [ ]:
import syft_client as sc

ds_client = sc.login_ds(email=DS_EMAIL)

## 4. Add DO as peer

In [ ]:
ds_client.add_peer(DO_EMAIL)

In [ ]:
ds_client.peers

## 5. Explore datasets

In [ ]:
datasets = ds_client.datasets.get_all(datasite=DO_EMAIL)
datasets

In [ ]:
dataset = datasets[-1]
dataset.describe()

## 6. Write analysis script

This script must match exactly what the DO registered for auto-approval.

In [ ]:
%%writefile main.py
import syft_client as sc

data_path = "syft://private/syft_datasets/Sales Dataset/sales.csv"
resolved_path = sc.resolve_path(data_path)

import pandas as pd

df = pd.read_csv(resolved_path)
total = (df["quantity"] * df["price_per_unit"]).sum()
print(f"Total Sales: {total}")

## 7. Submit job

The DO's auto-approval service will match this script and approve it automatically.

In [ ]:
ds_client.submit_python_job(
    user=DO_EMAIL,
    code_path="main.py",
    job_name="sales_analysis",
)

## 8. Monitor job status

Re-run this cell to check progress. The job should go from `pending` → `approved` → `done`.

In [ ]:
ds_client.jobs

## 9. View results

In [ ]:
job = ds_client.jobs[0]
print(f"Status: {job.status}")
job.stdout

## 10. Check your email

You should have received:
- **"Job Approved: sales_analysis"** — when the DO's auto-approval matched your script
- **"Job Completed: sales_analysis"** — when the job finished executing